# Executive Commerce Intelligence — Power BI + Tableau

This notebook is the interview-friendly walkthrough for the BI project. The production assets stay in normal source files: `prepare_bi_data.py` prepares governed exports, `power_bi/` contains the PBIP/PBIR + TMDL semantic model and report source, and `tableau/` contains the Tableau workbook source.

**Business question:** Where is marketplace value being created, where is customer experience breaking down, and what should leadership investigate first?

## 1. Data contract before dashboard design

The dashboards do not join raw order, item, payment and review tables directly. They consume validated exports from the upstream e-commerce SQL project, where reporting grain and reconciliation are already controlled. That prevents a visually polished dashboard from being built on duplicated revenue.

In [ ]:
from pathlib import Path
import json
import pandas as pd

project_dir = Path.cwd()
if project_dir.name != 'executive_commerce_bi':
    project_dir = project_dir / 'projects' / 'executive_commerce_bi'
data_dir = project_dir / 'data'
print('Project:', project_dir)
print('Expected governed export directory:', data_dir)

## 2. Build the governed BI exports

Run the upstream SQL project first, then execute `prepare_bi_data.py`. The prep script checks required columns and writes a manifest containing row counts, column names and SHA-256 hashes. Both Power BI and Tableau use these same outputs so metric logic is not duplicated between tools.

In [ ]:
# After running prepare_bi_data.py, inspect the generated contract.
manifest_path = data_dir / 'manifest.json'
if manifest_path.exists():
    manifest = json.loads(manifest_path.read_text())
    for filename, info in manifest['files'].items():
        print(f"{filename}: {info['rows']} rows | sha256={info['sha256'][:12]}…")
else:
    print('Run prepare_bi_data.py to create the governed exports and manifest.')

## 3. KPI layer

The retained commercial evidence is **98,199 commercial orders**, **94,983 unique customers**, **R$13.49M merchandise value**, **3.03% repeat customers** and **13.24% top-10 seller share**. Delivery/review analysis shows an average review score of **4.28/5 for on-time or early deliveries** versus **2.55/5 for late deliveries**.

The KPI dictionary makes the denominator and reporting grain explicit. Merchandise value is not described as profit, and the delivery/review relationship is presented as observational rather than causal.

In [ ]:
kpi_file = data_dir / 'executive_kpis.csv'
if kpi_file.exists():
    kpis = pd.read_csv(kpi_file)
    display(kpis.T.rename(columns={0: 'value'}))
else:
    print('Generate executive_kpis.csv before running this inspection cell.')

## 4. Dashboard story

The project uses four management views rather than a wall of charts:

1. **Executive Pulse** — scale, GMV, AOV and repeat behaviour.
2. **Customer & Category** — where marketplace value comes from.
3. **Delivery & Experience** — where operational friction aligns with poorer reviews.
4. **Marketplace Health** — seller concentration and payment behaviour.

The goal is that a reviewer can identify the management question first and then drill into supporting evidence.

## 5. Power BI implementation

The Power BI project is committed in source-control-friendly form using PBIP/PBIR and a TMDL semantic model. DAX measures define Commercial Orders, Unique Customers, Merchandise Value, Average Order Value, Repeat Customer %, seller concentration and analytical measures used by the report.

This makes the semantic layer inspectable in Git instead of presenting only a screenshot or opaque binary `.pbix`.

In [ ]:
power_bi_root = project_dir / 'power_bi'
for relative in [
    'ExecutiveCommerce.pbip',
    'ExecutiveCommerce.SemanticModel/definition/model.tmdl',
    'ExecutiveCommerce.Report/definition.pbir',
]:
    print(relative, '✓' if (power_bi_root / relative).exists() else 'missing')

## 6. Tableau implementation

The Tableau version consumes the same governed metric export. The `.twb` workbook source and calculation definitions are committed so the dashboard structure is reviewable in Git. The two tools deliberately share one KPI contract instead of containing separate hand-maintained business logic.

In [ ]:
tableau_root = project_dir / 'tableau'
print('Workbook source:', tableau_root / 'ExecutiveCommerce.twb')
print('Calculation pack:', tableau_root / 'calculations.md')

## 7. Interview discussion / limitations

I would use this project to discuss why trustworthy BI starts before the visual layer: grain, reconciliation, KPI definitions and repeatable data preparation come first. I would also explain why the same data contract is useful across Power BI and Tableau.

**Limitations:** the Olist dataset is historical; customer identifiers are anonymised; delivery/review findings do not establish causality; merchandise value is not profit. Power BI Desktop and Tableau Desktop/Public are required for the final interactive refresh/publish step, while GitHub retains the source-controlled model/workbook definitions and static portfolio preview.